# Measure the performance of the model before any quantization.

# Experiment 1: FP16 Baseline

Objective:
Measure the loading time, GPU memory usage, inference latency, throughput,
and perplexity of Llama-3.2-3B-Instruct in FP16 precision.

Note:
Before collecting measurements, the model was downloaded and cached locally.
Reported loading times exclude the initial network download.

In [ ]:
!pip install -U datasets huggingface_hub transformers evaluate

In [ ]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

Tue Jun 30 18:59:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from datasets import load_dataset
import time
import math


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    return tokenizer, model

In [ ]:
try:
    tokenizer, model
    print("Model already loaded, skipping.")
except NameError:
    print("Loading model...")
    start = time.time()
    tokenizer, model = load_model()
    end = time.time()
    loading_time = end - start
    print(f"Loading time: {loading_time:.2f} seconds")



Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading time: 28.94 seconds


In [ ]:
PROMPT = "Explain what a neural network is in simple terms."

inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    first_token_output = model.generate(
        **inputs,
        max_new_tokens=1,
        do_sample=False,
    )

torch.cuda.synchronize()
first_token_latency = time.time() - start

print(f"\nFirst token latency: {first_token_latency:.3f} seconds")

NUM_TOKENS = 200

torch.cuda.synchronize()
start = time.time()

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=NUM_TOKENS,
        do_sample=False,
    )

torch.cuda.synchronize()
generation_time = time.time() - start

prompt_length = inputs["input_ids"].shape[1]
generated_tokens = output.shape[1] - prompt_length

tokens_per_second = generated_tokens / generation_time


decoded = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"\nModel output:\n{decoded}")



[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



First token latency: 1.008 seconds


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Model output:
Explain what a neural network is in simple terms. A neural network is a computer system that is modeled after the human brain. It's made up of layers of interconnected nodes or "neurons" that process and transmit information. Just like how our brains process information, a neural network processes information by passing it through these layers, allowing it to learn and make decisions.
In simple terms, a neural network is a computer program that is designed to mimic the way our brains work. It's made up of many interconnected nodes, or "neurons", that process and transmit information. Just like how our brains process information, a neural network processes information by passing it through these layers, allowing it to learn and make decisions.
Here's a simple analogy to help illustrate how a neural network works:
Imagine you're trying to recognize a picture of a cat. A neural network would look at the picture and break it down into smaller parts, like the shape of the ear

In [ ]:

from tqdm.auto import tqdm

# Load only a small subset for benchmarking
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test[:50]"
)

model.eval()

loss_sum = 0.0
token_count = 0

with torch.no_grad():

    for sample in tqdm(dataset):

        text = sample["text"].strip()

        if len(text) < 20:
            continue

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        outputs = model(
            **inputs,
            labels=inputs["input_ids"]
        )

        num_tokens = inputs["input_ids"].size(1)

        loss_sum += outputs.loss.item() * num_tokens
        token_count += num_tokens

        # immediately free memory
        del inputs
        del outputs
        torch.cuda.empty_cache()

avg_loss = loss_sum / token_count
perplexity = torch.exp(torch.tensor(avg_loss))



  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
from datasets import load_dataset

boolq = load_dataset(
    "google/boolq",
    split="validation[:100]"
)

print(boolq[0])

{'question': 'does ethanol take more energy make that produces', 'answer': False, 'passage': "All biomass goes through at least some of these steps: it needs to be grown, collected, dried, fermented, distilled, and burned. All of these steps require resources and an infrastructure. The total amount of energy input into the process compared to the energy released by burning the resulting ethanol fuel is known as the energy balance (or ``energy returned on energy invested''). Figures compiled in a 2007 report by National Geographic Magazine point to modest results for corn ethanol produced in the US: one unit of fossil-fuel energy is required to create 1.3 energy units from the resulting ethanol. The energy balance for sugarcane ethanol produced in Brazil is more favorable, with one unit of fossil-fuel energy required to create 8 from the ethanol. Energy balance estimates are not easily produced, thus numerous such reports have been generated that are contradictory. For instance, a separ

In [ ]:
from tqdm.auto import tqdm
import torch

model.eval()

correct = 0
evaluated = 0

for example in tqdm(boolq):

    messages = [
        {
            "role": "user",
            "content": f"""Passage:
{example["passage"]}

Question:
{example["question"]}

Answer only with Yes or No."""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=3,
            do_sample=False,
            temperature=None,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip().lower()

    if answer.startswith("yes"):
        prediction = True
    elif answer.startswith("no"):
        prediction = False
    else:
        del inputs
        del outputs
        torch.cuda.empty_cache()
        continue

    if prediction == example["answer"]:
        correct += 1

    evaluated += 1

    del inputs
    del outputs
    torch.cuda.empty_cache()

accuracy = correct / evaluated

print(f"Accuracy: {accuracy*100:.2f}% ({correct}/{evaluated})")

  0%|          | 0/100 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_toke

Accuracy: 76.77% (76/99)


In [ ]:
print("\n" + "=" * 55)
print("           FP16 BENCHMARK RESULTS")
print("=" * 55)

print(f"Model                 : {MODEL_NAME}")
print(f"Precision             : FP16")
print(f"GPU                   : {torch.cuda.get_device_name(0)}")

print("-" * 55)

print(f"Model Loading Time    : {loading_time:.2f} s")
print(f"VRAM Used             : {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM Total            : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

print("-" * 55)

print(f"First Token Latency   : {first_token_latency:.3f} s")
print(f"Generation Time       : {generation_time:.2f} s")
print(f"Generated Tokens      : {generated_tokens}")
print(f"Throughput            : {tokens_per_second:.2f} tokens/s")

print("-" * 55)

print(f"Perplexity            : {perplexity.item():.2f}")
print(f"BoolQ Accuracy        : {accuracy*100:.2f}%")
print(f"Samples Evaluated     : {evaluated}")

print("=" * 55)


           FP16 BENCHMARK RESULTS
Model                 : meta-llama/Llama-3.2-3B-Instruct
Precision             : FP16
GPU                   : Tesla T4
-------------------------------------------------------
Model Loading Time    : 28.94 s
VRAM Used             : 6.43 GB
VRAM Total            : 15.64 GB
-------------------------------------------------------
First Token Latency   : 1.008 s
Generation Time       : 14.53 s
Generated Tokens      : 200
Throughput            : 13.76 tokens/s
-------------------------------------------------------
Perplexity            : 17.05
BoolQ Accuracy        : 76.77%
Samples Evaluated     : 99


In [ ]:
fp16_results = {
    "Precision": "FP16",
    "VRAM (GB)": torch.cuda.memory_allocated()/1e9,
    "Latency (s)": first_token_latency,
    "Tokens/s": tokens_per_second,
    "Perplexity": perplexity.item(),
    "Accuracy (%)": accuracy*100,
}

In [ ]:
RESULTS_DIR = "/content/drive/MyDrive/ExperimentalProjects/Quantization/results"

In [ ]:
import pandas as pd
import os

fp16_results = {
    "Precision": "FP16",
    "VRAM (GB)": torch.cuda.memory_allocated()/1e9,
    "Latency (s)": first_token_latency,
    "Tokens/s": tokens_per_second,
    "Perplexity": perplexity.item(),
    "Accuracy (%)": accuracy*100,
}

df = pd.DataFrame([fp16_results])

os.makedirs(RESULTS_DIR, exist_ok=True)

df.to_csv(
    os.path.join(RESULTS_DIR, "fp16_results.csv"),
    index=False
)

print("Saved!")

Saved!
